<a href="https://colab.research.google.com/github/Andresflorezdev/Recognition/blob/main/Reconocimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cómo funciona

### Celda 1 (ejecutar una sola vez)
Crea un parche para que TensorFlow pueda cargar el modelo `.h5`.

¿Por qué?  
Porque el modelo fue entrenado con una versión antigua que usa el parámetro `groups=1` en `DepthwiseConv2D`, y la versión actual no lo reconoce.

Esta celda elimina ese parámetro automáticamente.

**Solo debes ejecutarla otra vez si reinicias el runtime.**

---

### Celda 2 (ejecutar las veces que quieras)
Carga el modelo y hace la predicción de la imagen.

Puedes ejecutarla varias veces porque el parche ya quedó cargado en memoria.

Para probar otra imagen, solo cambia:

```python
image = Image.open("/content/tu_imagen.jpg").convert("RGB")



Desinstalacion

In [ ]:
!pip uninstall -y jax jaxlib

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2


Instalacion de Keras

In [ ]:
!pip install tf-keras==2.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.5 MB/s eta 0:00:00
  Attempting uninstall: tf-keras
    Found existing installation: tf_keras 2.20.0
    Uninstalling tf_keras-2.20.0:
      Successfully uninstalled tf_keras-2.20.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, but you have tf-keras 2.16.0 which is incompatible.


## TF_USE_LEGACY_KERAS

```python
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
¿Para qué sirve?

Le indica a TensorFlow que use la versión antigua de Keras (legacy).

¿Por qué se usa?

Porque el modelo .h5 fue creado con una versión vieja y puede dar errores con Keras nuevo (como DepthwiseConv2D o groups).

Importante

Debe ir antes de importar TensorFlow.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

Verificar Version

In [ ]:
import tensorflow as tf
print(tf.__version__)

2.16.2


Celda 1 (solo una vez)

Esta celda parcha/modifica TensorFlow temporalmente para que pueda entender tu archivo .h5.

```
# Esto tiene formato de código
import tensorflow as tf

class CustomDepthwiseConv2D(tf.keras.layers.DepthwiseConv2D):
    def __init__(self, *args, **kwargs):
        kwargs.pop("groups", None)
        super().__init__(*args, **kwargs)
```

Qué hace

Tu modelo fue entrenado con una versión vieja de Keras/TensorFlow donde DepthwiseConv2D guardaba este parámetro:

groups=1

Pero tu TensorFlow actual no lo reconoce.

Entonces esta celda crea una versión personalizada de esa capa que dice:

"Si aparece groups, ignóralo y sigue cargando."

Por qué solo una vez

Porque una vez que ejecutas esta celda:

la clase queda registrada en memoria
TensorFlow ya sabe cómo interpretar el modelo

No necesitas redefinirla cada vez.

Solo tendrías que volver a correrla si:

reiniciaste el runtime
cerraste Colab
hiciste "Factory reset runtime"

In [ ]:
from tensorflow.keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Define a custom DepthwiseConv2D class to handle the 'groups' argument (if needed for this cell too)
class CustomDepthwiseConv2D(tf.keras.layers.DepthwiseConv2D):
    def __init__(self, *args, **kwargs):
        kwargs.pop("groups", None)
        super().__init__(*args, **kwargs)

# Load the model
model = tf.keras.models.load_model(
    "/content/keras_model.h5",
    compile=False,
    safe_mode=False,
    custom_objects={'DepthwiseConv2D': CustomDepthwiseConv2D}
)

# Load the labels
class_names = open("/content/labels.txt", "r").readlines()

Celda 2 (las veces que quieras)

Esta ya hace la predicción:


```
# Esto tiene formato de código

model = tf.keras.models.load_model(...)
```

Luego:

carga imagen
la redimensiona
la normaliza
predice
muestra resultado
Por qué sí puedes correrla varias veces

Porque ya no está "enseñándole" a TensorFlow cómo leer el modelo.

Solo está usándolo.

Es como esto:

Celda 1: instalar el traductor

Celda 2: usar el traductor

Instalas una vez, usas muchas.

In [ ]:
# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open("/content/Image-foto.jpeg").convert("RGB")

# resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

# turn the image into a numpy array
image_array = np.asarray(image)

# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

# Load the image into the array
data[0] = normalized_image_array

# Predicts the model
prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

1/1 [==============================] - 0s 39ms/step
Class: cara normal
Confidence Score: 0.99800974
